In [1]:
import pandas as pd
import re

print("⏳ Memulai proses penyatuan dan pembersihan dataset Exigen...")

# 1. Tentukan Path
path_lokasi_revisi = "../../../data/dataset_tiket_lengkap_lokasi_revisi.csv"
path_revisi        = "../../../data/dataset_tiket_lengkap_dupe_revisi.csv"
path_utama         = "../../../data/dataset_tiket_lengkap_gedung.csv"
path_final         = "../../../data/dataset_tiket_master_bersih.csv"

# 2. Baca Semua Dataset
df1 = pd.read_csv(path_lokasi_revisi, sep='|', engine='python', on_bad_lines='skip')
df2 = pd.read_csv(path_revisi, sep='|', engine='python', on_bad_lines='skip')
df3 = pd.read_csv(path_utama, sep='|', engine='python', on_bad_lines='skip')

# 3. Gabungkan (Concat) Menjadi Satu DataFrame
df_merged = pd.concat([df1, df2, df3], ignore_index=True)
print(f"🔹 Total baris setelah digabung: {len(df_merged)}")

# Buang baris yang keluhannya kosong (NaN) untuk mencegah error Regex
kolom_wajib = ['teks_keluhan_awam', 'lokasi_gedung', 'lokasi_lantai', 'lokasi_zona']
df_merged = df_merged.dropna(subset=kolom_wajib).reset_index(drop=True)

# 4. Hapus Duplikat Identik
df_merged = df_merged.drop_duplicates(subset=['teks_keluhan_awam']).reset_index(drop=True)
print(f"🔹 Total baris setelah hapus duplikat teks: {len(df_merged)}")

# =====================================================================
# 5. PERBAIKAN 1: Hapus Teks Stuttering (Gedung-Gedung A -> Gedung A)
# =====================================================================
def hapus_kata_berulang(teks):
    # Regex ini mencari kata yang diketik berulang secara berurutan dan menyisakan satu saja
    return re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', str(teks), flags=re.IGNORECASE)

# Terapkan ke teks keluhan awam dan label lokasi gedung
df_merged['teks_keluhan_awam'] = df_merged['teks_keluhan_awam'].apply(hapus_kata_berulang)
df_merged['lokasi_gedung'] = df_merged['lokasi_gedung'].apply(hapus_kata_berulang)


# 6. Hapus "Label Halusinasi LLM"
kondisi_tanpa_kata_gedung = ~df_merged['teks_keluhan_awam'].str.contains(r'\b(gedung|gdng|gd\.?|tower|twr|blok|blk|lobby|lobi|area)\b', case=False, na=False)
kondisi_label_terisi = ~df_merged['lokasi_gedung'].isin(['Unknown', 'Tidak Disebutkan', '-'])
baris_halusinasi = kondisi_tanpa_kata_gedung & kondisi_label_terisi

df_clean = df_merged[~baris_halusinasi].reset_index(drop=True)
print(f"✂️ Dihapus {baris_halusinasi.sum()} baris halusinasi LLM.")


# =====================================================================
# 7. PERBAIKAN 2: Harmonisasi Zona ("Barat" vs "Zona Barat")
# =====================================================================
def harmonisasi_zona(val):
    if val in ['Unknown', '-', 'Tidak Disebutkan']:
        return "Unknown"
    # Bersihkan kata 'zona' terlebih dahulu agar netral, lalu pasang ulang dengan rapi
    val_clean = str(val).lower().replace('zona', '').strip()
    return f"Zona {val_clean.capitalize()}" if val_clean else "Unknown"

df_clean['lokasi_zona'] = df_clean['lokasi_zona'].apply(harmonisasi_zona)


# =====================================================================
# 8. PERBAIKAN 3: Harmonisasi Lantai ("1" vs "Lantai 1")
# =====================================================================
def harmonisasi_lantai(val):
    val_clean = str(val).lower().replace('lantai', '').strip()
    if val_clean.isdigit():
        return f"Lantai {val_clean}"
    if val_clean and val_clean not in ['nan', '-']:
        return f"Lantai {val_clean.capitalize()}"
    return "Lantai Unknown"

df_clean['lokasi_lantai'] = df_clean['lokasi_lantai'].apply(harmonisasi_lantai)


# 9. Simpan ke dalam 1 File Utama (Golden Dataset)
print(f"\n📊 Total baris FINAL yang siap masuk training: {len(df_clean)}")
df_clean.to_csv(path_final, sep='|', index=False)
print(f"✅ Sukses! Golden Dataset berhasil disimpan di: {path_final}")

⏳ Memulai proses penyatuan dan pembersihan dataset Exigen...
🔹 Total baris setelah digabung: 3379
🔹 Total baris setelah hapus duplikat teks: 2933
✂️ Dihapus 642 baris halusinasi LLM.

📊 Total baris FINAL yang siap masuk training: 2291
✅ Sukses! Golden Dataset berhasil disimpan di: ../../../data/dataset_tiket_master_bersih.csv


C:\Users\ryama\AppData\Local\Temp\ipykernel_34796\2814809585.py:42: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  kondisi_tanpa_kata_gedung = ~df_merged['teks_keluhan_awam'].str.contains(r'\b(gedung|gdng|gd\.?|tower|twr|blok|blk|lobby|lobi|area)\b', case=False, na=False)
